In [1]:
import numpy as np
from statsmodels.stats.multitest import multipletests
import pandas as pd
from data_stream.real_data_stream import data_id_2name
import os

## This part is for Benjamini-Yekutieli (BY) to adjust the p-value. Due to the most all the orignial p-values are smaller than 10^-5, our conclusion of the experiment would not be affect.

In [3]:
os.makedirs("results/RQ2/performance/adjustment/",exist_ok=True)

In [4]:
datasets=[data_id_2name(i) for i in range(1,23,1)]

In [5]:
# For RQ1

features=["age","lt","nuc","rexp","ndev"]

for feature in features:
    p_values=  pd.read_csv(f"results/RQ1/table/RQ11/{feature}_statistics.csv")["Wilcoxon"].to_numpy()
    original_reject = p_values<0.05
    # Use Benjamini-Yekutieli Adjustment
    reject, pvals_corrected, _, _ = multipletests(
        pvals=p_values,
        alpha=0.05,          # alhpa default is 0.05
        method='fdr_by'      # use Benjamini-Yekutieli (BY)
    )
    is_same=original_reject==reject
    # Output
    # print("original p-value:", p_values)
    # print("after Benjamini adjustment:", pvals_corrected)
    # print(f"Original Reject:{original_reject}")
    # print("New reject:", reject)
    # print(f"Is same: {is_same}")

    output=pd.DataFrame(np.array([p_values,pvals_corrected,original_reject,reject,is_same]).T,
                 index=datasets,columns=["original p-value","after Benjamini adjustment","Original Reject","New reject","Is same"])
    output[["Original Reject","New reject","Is same"]]=output[["Original Reject","New reject","Is same"]].astype(bool)
    output.to_csv(f"results/RQ1/table/RQ11/{feature}_adjustment_pvalue.csv")


In [7]:
# For RQ2

metrics=["avg_R0","avg_R1","avg_Gmean","avg_mcc","avg_precision","avg_f1_score","avg_acc"]
models=["humla-effort-1","humla-effort-auto","odasc","oob","pbsa"]



for model in models:
    p_values_df=pd.read_csv(f"results/RQ2/performance/diff/{model}_p_value.csv")
    adjust_reject=[]
    adjust_pvalue=[]
    for metric in metrics:
        p_values= p_values_df[metric].to_numpy()[:-1]
        original_reject = p_values<0.05
        # Use Benjamini-Yekutieli Adjustment
        reject, pvals_corrected, _, _ = multipletests(
            pvals=p_values,
            alpha=0.05,          # alhpa default is 0.05
            method='fdr_by'      # use Benjamini-Yekutieli (BY)
        )
        is_same=original_reject==reject
        # print(is_same)
        adjust_reject.append(reject)
        adjust_pvalue.append(pvals_corrected)
    pd.DataFrame(np.array(adjust_pvalue).T,index=datasets,columns=metrics).to_csv(f"results/RQ2/performance/adjustment/{model}_ad_pvalue.csv")
    pd.DataFrame(np.array(adjust_reject).T,index=datasets,columns=metrics).to_csv(f"results/RQ2/performance/adjustment/{model}_ad_reject.csv")
